In [ ]:
%pip install langchain langchain-google-vertexai jsonlines

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.7/408.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.5/142.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: google-cloud-storage
    Found existing installation: google-cloud-storage 2.8.0
    Uninstalling google

In [2]:
# Initialize cloud storage
from google.cloud import storage

BUCKET_NAME = "cloud-ai-platform-e215f7f7-a526-4a66-902d-eb69384ef0c4"
storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

In [3]:
# Loop over all of the abstracts and split them into chunks suitable for creating an embedding
from langchain_google_vertexai import VertexAIEmbeddings
import jsonlines
import json

# Initialize the text splitter
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 20
)

# Initialize the a specific Embeddings Model version
embeddings = VertexAIEmbeddings(model_name="text-embedding-004")

chunk_id = 0
chunk_to_doc = {} # A lookup so we know where a ANN result points to.
embedding_jsonl = bucket.blob('semantic-search/index/index.json')

with embedding_jsonl.open("w") as f:
  with jsonlines.open('dataset.json') as reader:
    for doc in reader:
      split_documents = text_splitter.create_documents([doc['abstract']])
      for chunk in split_documents:
        chunk_id += 1
        content = chunk.page_content
        chunk_to_doc[chunk_id] = { "purl": doc['id'], "content": content }
        vector = embeddings.embed_query(content)
        f.write(json.dumps({"id": chunk_id, "embedding": vector}) + "\n")

stored_chunks = bucket.blob('semantic-search/chunk_to_doc.json')
with stored_chunks.open("w") as f:
  f.write(json.dumps(chunk_to_doc))

In [ ]:
# Query
from google.cloud import aiplatform_v1

# Set variables for the current deployed index.
API_ENDPOINT="430778453.us-central1-768608702519.vdb.vertexai.goog"
INDEX_ENDPOINT="projects/768608702519/locations/us-central1/indexEndpoints/5525191065109266432"
DEPLOYED_INDEX_ID="open_access_deploy_1730924393870"

# Configure Vector Search client
client_options = {
  "api_endpoint": API_ENDPOINT
}
vector_search_client = aiplatform_v1.MatchServiceClient(
  client_options=client_options,
)

# Build FindNeighborsRequest object
datapoint = aiplatform_v1.IndexDatapoint(
  feature_vector="<FEATURE_VECTOR>"
)

query = aiplatform_v1.FindNeighborsRequest.Query(
  datapoint=datapoint,

  # The number of nearest neighbors to be retrieved
  neighbor_count=10
)
request = aiplatform_v1.FindNeighborsRequest(
  index_endpoint=INDEX_ENDPOINT,
  deployed_index_id=DEPLOYED_INDEX_ID,
  # Request can have multiple queries
  queries=[query],
  return_full_datapoint=False,
)

# Execute the request
response = vector_search_client.find_neighbors(request)

# Handle the response
print(response)

In [13]:
# Run a test query
from google.cloud import aiplatform
DEPLOYED_INDEX_ID = 'open_access_deploy_1730924393870' # @param {type:"string"}
INDEX_ENDPOINT_ID = '5525191065109266432' # @param {type:"string"}
PROJECT_ID = "sul-ai-sandbox"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
QUERY_TEXT = "Health care outside of the united states" # @param {type:"string"}
aiplatform.init(project=PROJECT_ID, location=LOCATION)
my_index_endpoint = aiplatform.MatchingEngineIndexEndpoint(
    index_endpoint_name=INDEX_ENDPOINT_ID
)

vector = embeddings.embed_query(QUERY_TEXT)

responses = my_index_endpoint.find_neighbors(
    deployed_index_id = DEPLOYED_INDEX_ID,
    queries = [vector],
    num_neighbors = 10
)
response = responses[0]
print(response)
for neighbor in response:
  print(chunk_to_doc[int(neighbor.id)], neighbor.distance)

[MatchNeighbor(id='16907', distance=0.5331038236618042, sparse_distance=None, feature_vector=[], crowding_tag='0', restricts=[], numeric_restricts=[], sparse_embedding_values=[], sparse_embedding_dimensions=[]), MatchNeighbor(id='6596', distance=0.5273138880729675, sparse_distance=None, feature_vector=[], crowding_tag='0', restricts=[], numeric_restricts=[], sparse_embedding_values=[], sparse_embedding_dimensions=[]), MatchNeighbor(id='6598', distance=0.519698977470398, sparse_distance=None, feature_vector=[], crowding_tag='0', restricts=[], numeric_restricts=[], sparse_embedding_values=[], sparse_embedding_dimensions=[]), MatchNeighbor(id='3346', distance=0.5175025463104248, sparse_distance=None, feature_vector=[], crowding_tag='0', restricts=[], numeric_restricts=[], sparse_embedding_values=[], sparse_embedding_dimensions=[]), MatchNeighbor(id='11358', distance=0.5150790810585022, sparse_distance=None, feature_vector=[], crowding_tag='0', restricts=[], numeric_restricts=[], sparse_em

In [8]:
print(chunk_to_doc[3083])

{'purl': 'fr662qb6767', 'content': 'Our cross-sectional findings support a growing body of research linking CRP to alterations in reward processing regions of the brain. Most previous studies have focused on functional Magnetic Resonance Imaging in examining neural responses to reward stimuli. Our structural findings add to this body of work, given that brain structure is posited to underlie function. The moderating effect of ELS on the association between CRP and NAcc GMV support predictions from the neuroimmune network hypothesis and indicates that childhood deprivation can affect the relation between inflammation and brain structure in reward processing regions.'}
